# NSMC 영화 리뷰 감정분석 리팩토링 실험

이 노트북은 실험 실행과 결과 비교만 담당합니다. 데이터 전처리, 토크나이저, 모델, 학습/평가 로직은 기능별 Python 파일로 분리했습니다.

## 1. 모듈 로드

파일명이 `01_`, `02_`처럼 숫자로 시작하므로 `importlib`로 모듈을 불러옵니다.

In [ ]:
from pathlib import Path
import importlib
import sys

MODULE_DIR = Path.cwd()
if not (MODULE_DIR / "01_data_preprocessing.py").exists():
    MODULE_DIR = Path("/Users/park_jinyong/Desktop/aiffel/AIFFEL_quest_rs/GoingDeeper/GD0102/legacy")

if str(MODULE_DIR) not in sys.path:
    sys.path.append(str(MODULE_DIR))

data_module = importlib.import_module("01_data_preprocessing")
experiment_module = importlib.import_module("04_experiment")

load_and_preprocess_data = data_module.load_and_preprocess_data
ExperimentConfig = experiment_module.ExperimentConfig
seed_everything = experiment_module.seed_everything
run_experiment = experiment_module.run_experiment

## 2. 데이터 준비

NSMC train/test 데이터를 로드하고, 결측치 제거, 중복 제거, train/validation split을 수행합니다.

In [ ]:
config = ExperimentConfig()
seed_everything(config.SEED)

# 기본값은 기존 노트북의 NSMC data 경로를 사용합니다.
# 다른 위치에 ratings_train.txt, ratings_test.txt가 있다면 DATA_DIR에 경로를 넣으세요.
DATA_DIR = None

train_set, val_set, test_set = load_and_preprocess_data(
    data_dir=DATA_DIR,
    seed=config.SEED,
)

print("훈련 데이터셋 개수:", len(train_set))
print("검증 데이터셋 개수:", len(val_set))
print("테스트 데이터셋 개수:", len(test_set))

## 3. KoNLPy Mecab 실험

기존 통합 실험 블록의 기본 설정과 동일하게 GRU 모델을 학습하고 평가합니다.

In [ ]:
mecab_config = ExperimentConfig(
    TOKENIZER_TYPE="KoNLPy",
    KONLPY_TOKENIZER="Mecab",
    KONLPY_VOCAB_SIZE=10000,
    EMBEDDING_DIM=128,
    HIDDEN_DIM=128,
    N_LAYERS=1,
    DROPOUT=0.6,
    EPOCHS=20,
    LEARNING_RATE=0.001,
    BATCH_SIZE=128,
    MAX_LEN=40,
    PATIENCE=3,
    WEIGHT_DECAY=1e-5,
    SEED=42,
    MODEL_SAVE_PATH="best_model_mecab.pth",
)

seed_everything(mecab_config.SEED)
result_mecab = run_experiment(mecab_config, train_set, val_set, test_set)
result_mecab

## 4. SentencePiece BPE 실험

In [ ]:
sp_bpe_config = ExperimentConfig(
    TOKENIZER_TYPE="SentencePiece",
    SP_MODEL_TYPE="bpe",
    SP_VOCAB_SIZE=10000,
    EMBEDDING_DIM=128,
    HIDDEN_DIM=128,
    N_LAYERS=1,
    DROPOUT=0.6,
    EPOCHS=20,
    LEARNING_RATE=0.001,
    BATCH_SIZE=128,
    MAX_LEN=40,
    PATIENCE=3,
    WEIGHT_DECAY=1e-5,
    SEED=42,
    MODEL_SAVE_PATH="best_model_sp_bpe.pth",
)

seed_everything(sp_bpe_config.SEED)
result_sp_bpe = run_experiment(sp_bpe_config, train_set, val_set, test_set)
result_sp_bpe

## 5. SentencePiece Unigram 실험

필요할 때 실행합니다. 기존 실험 조건을 유지하되 `SP_MODEL_TYPE`만 `unigram`으로 바꿉니다.

In [ ]:
sp_unigram_config = ExperimentConfig(
    TOKENIZER_TYPE="SentencePiece",
    SP_MODEL_TYPE="unigram",
    SP_VOCAB_SIZE=10000,
    EMBEDDING_DIM=128,
    HIDDEN_DIM=128,
    N_LAYERS=1,
    DROPOUT=0.6,
    EPOCHS=20,
    LEARNING_RATE=0.001,
    BATCH_SIZE=128,
    MAX_LEN=40,
    PATIENCE=3,
    WEIGHT_DECAY=1e-5,
    SEED=42,
    MODEL_SAVE_PATH="best_model_sp_unigram.pth",
)

# 오래 걸리면 아래 줄만 주석 처리하고 필요한 실험만 실행하세요.
seed_everything(sp_unigram_config.SEED)
result_sp_unigram = run_experiment(sp_unigram_config, train_set, val_set, test_set)
result_sp_unigram

## 6. 결과 비교

In [ ]:
results = {
    "KoNLPy-Mecab": result_mecab,
    "SentencePiece-BPE": result_sp_bpe,
    "SentencePiece-Unigram": result_sp_unigram,
}

results

## 리팩토링 매핑

- 데이터 로드, 결측치 제거, 중복 제거, train/validation split, SentencePiece corpus 저장: `01_data_preprocessing.py`
- SentencePiece 학습/로드/인코딩, KoNLPy Mecab/Okt, vocab 생성, padding/truncation: `02_train_tokenizer.py`
- `SentimentGRU` 모델 클래스: `03_model.py`
- `ExperimentConfig`, Dataset/DataLoader, train loop, validation, test evaluation, loss, optimizer, early stopping, checkpoint, accuracy: `04_experiment.py`